# Latent Fusion — Demonstração Completa

**Rodrigo Banin Ferraz de Camargo** — Instituto de Computação, UNICAMP  
Orienta  o: Prof. Dr. Allan Mariano de Souza

---

Este notebook demonstra o pipeline completo de fusão multimodal para predição financeira:

1. Embeddings textuais de notícias (SentenceTransformer)
2. Detecção de regimes de mercado (HMM + volatilidade)
3. Intensidade de eventos (Hawkes-style)
4. Estratégias de trading (8 implementações)
5. Modelos ML treinados por perfil de investidor
6. Backtest engine completo
7. Paper trading ao vivo

Todos os gráficos seguem o padrão dark-mode publication-ready do projeto.

In [ ]:
import sys, os, json, time, warnings
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)

BG    = '#0d0d1a'
PANEL = '#1a1a2e'
GOLD  = '#D4A843'
GREEN = '#00E676'
RED   = '#FF1744'
CYAN  = '#18FFFF'
WHITE = '#AAAAAA'
PURPLE = '#CE93D8'
ORANGE = '#FF9800'
BLUE  = '#42A5F5'

IMAGES = _root / 'images/showcase'
IMAGES.mkdir(parents=True, exist_ok=True)

def dark_fig(figsize=(14,7)):
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(PANEL)
    return fig, ax

def dark_save(fig, name):
    fig.savefig(IMAGES / name, dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
    plt.show()

print('Ambiente configurado. Raiz:', _root)

## 1. Dados — Carteira Híbrida (NASDAQ + Crypto + B3)

Dataset unificado com 44 ativos de 3 mercados distintos.

In [ ]:
from src.backtest.engine import BacktestEngine, BacktestConfig
from src.backtest.visualization import BG as _bg, PANEL as _pn, GOLD as _gd

has_full_data = (_root / 'data/lse_market_data/combined_1d.parquet').exists()
prices_path = _root / ('data/lse_market_data/combined_1d.parquet' if has_full_data else 'sample_data/prices/combined_1d.parquet')
prices = pd.read_parquet(prices_path)
prices['timestamp'] = pd.to_datetime(prices['timestamp']).dt.tz_localize(None).dt.normalize()

pivot = prices.pivot_table(index='timestamp', columns='symbol', values='close')
pivot = pivot.ffill().bfill()
returns = pivot.pct_change().fillna(0)

n_ativos = len(pivot.columns)
n_dias = len(pivot)
data_inicio = str(pivot.index[0].date())
data_fim = str(pivot.index[-1].date())

print(f'Período: {data_inicio} → {data_fim}')
print(f'Dias: {n_dias}  |  Ativos: {n_ativos}')
print(f'Fontes: {prices["asset_group"].unique().tolist() if "asset_group" in prices.columns else "sample"}')
display(returns.describe().round(4).iloc[:, :8])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 7))
fig.patch.set_facecolor(BG)

ax = axes[0]
ax.set_facecolor(PANEL)
cum = (1.0 + returns).cumprod()
for col in cum.columns[:12]:
    ax.plot(cum.index, cum[col].values, lw=0.8, alpha=0.5, label=col)
ax.set_title('Crescimento Acumulado — Top 12 Ativos', color='white', fontsize=14, fontweight='bold')
ax.set_xlabel('Data', color='white', fontsize=12)
ax.set_ylabel('Crescimento (Base=1)', color='white', fontsize=12)
ax.tick_params(colors='white', labelsize=10)
ax.grid(True, alpha=0.12)
ax.legend(fontsize=8, facecolor=PANEL, edgecolor='white', labelcolor='white', ncol=2)

ax = axes[1]
ax.set_facecolor(PANEL)
vol = returns.rolling(20).std().iloc[-1].sort_values() * np.sqrt(252)
colors = [GREEN if v < 0.5 else GOLD if v < 0.8 else RED for v in vol.values]
ax.barh(range(len(vol)), vol.values, color=colors, height=0.7)
ax.set_yticks(range(len(vol)))
ax.set_yticklabels(vol.index, fontsize=9, color='white')
ax.set_title('Volatilidade Anualizada (últimos 20 dias)', color='white', fontsize=14, fontweight='bold')
ax.set_xlabel('Volatilidade', color='white', fontsize=12)
ax.tick_params(colors='white', labelsize=10)
ax.grid(True, alpha=0.12, axis='x')

fig.suptitle('Visão Geral do Universo de Ativos', color='white', fontsize=16, fontweight='bold', y=1.01)
fig.tight_layout()
dark_save(fig, '01_universe_overview.png')

## 2. Embeddings Textuais — SentenceTransformer

Notícias financeiras são codificadas em vetores de 384 dimensões usando `all-MiniLM-L6-v2`.
PCA reduz para 32 componentes. O drift entre embeddings consecutivos mede a chegada de informação nova.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

has_text_cache = (_root / 'cache/text/top50_daily_embeddings.npy').exists()
if has_text_cache:
    emb_cache = _root / 'cache/text'
else:
    emb_cache = _root / 'sample_data/cache/text'

emb_all = np.load(emb_cache / 'top50_daily_embeddings.npy')
meta_sp = pd.read_csv(emb_cache / 'top50_daily_metadata.csv')
meta_sp['date'] = pd.to_datetime(meta_sp['date'], errors='coerce').dt.tz_localize(None).dt.normalize()
meta_sp = meta_sp.dropna(subset=['date'])

nasdaq_tickers = set(prices[prices['asset_group'] == 'nasdaq']['symbol'].unique()) if 'asset_group' in prices.columns else set(meta_sp['ticker'].unique())
sp_tickers = sorted(nasdaq_tickers & set(meta_sp['ticker'].unique()))
print(f'Tickers com embeddings: {len(sp_tickers)}')

pca_emb = PCA(n_components=2, random_state=42)
sample_embs = emb_all[meta_sp['ticker'].isin(sp_tickers[:5])][:1000]
sample_labels = meta_sp[meta_sp['ticker'].isin(sp_tickers[:5])]['ticker'].values[:1000]
coords = pca_emb.fit_transform(sample_embs)

fig, ax = dark_fig((12, 8))
for ticker in sp_tickers[:5]:
    mask = sample_labels == ticker
    ax.scatter(coords[mask, 0], coords[mask, 1], s=3, alpha=0.6, label=ticker)
ax.set_title(f'Embeddings Textuais — PCA 2D (Top 5 tickers, {len(coords)} amostras)', color='white', fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca_emb.explained_variance_ratio_[0]:.1%})', color='white', fontsize=12)
ax.set_ylabel(f'PC2 ({pca_emb.explained_variance_ratio_[1]:.1%})', color='white', fontsize=12)
ax.tick_params(colors='white', labelsize=10)
ax.grid(True, alpha=0.12)
ax.legend(fontsize=11, facecolor=PANEL, edgecolor='white', labelcolor='white')
dark_save(fig, '02_embeddings_pca.png')

In [ ]:
if len(sp_tickers) >= 2:
    t1, t2 = sp_tickers[0], sp_tickers[1]
    mask1 = meta_sp['ticker'] == t1
    mask2 = meta_sp['ticker'] == t2
    emb1 = emb_all[mask1.values]
    emb2 = emb_all[mask2.values]
    dates1 = meta_sp[mask1.values]['date'].values
    dates2 = meta_sp[mask2.values]['date'].values

    drift1 = np.linalg.norm(np.diff(emb1, axis=0, prepend=emb1[:1]), axis=1)
    drift2 = np.linalg.norm(np.diff(emb2, axis=0, prepend=emb2[:1]), axis=1)

    p95 = np.percentile(np.concatenate([drift1, drift2]), 95)

    fig, axes = plt.subplots(2, 1, figsize=(18, 8))
    fig.patch.set_facecolor(BG)

    for ax, (ticker, dates, drift) in zip(axes, [(t1, dates1, drift1), (t2, dates2, drift2)]):
        ax.set_facecolor(PANEL)
        ax.plot(dates, drift, color=CYAN, lw=0.8, alpha=0.7)
        ax.axhline(p95, color=RED, ls='--', lw=1, alpha=0.6, label=f'P95 = {p95:.3f}')
        events = drift >= p95
        ax.scatter(dates[events], drift[events], color=GOLD, s=20, alpha=0.8, zorder=5, label=f'Eventos (n={events.sum()})')
        ax.set_title(f'Drift de Embedding — {ticker}', color='white', fontsize=13, fontweight='bold')
        ax.set_ylabel('||E_t - E_{t-1}||_2', color='white', fontsize=12)
        ax.tick_params(colors='white', labelsize=10)
        ax.grid(True, alpha=0.12)
        ax.legend(fontsize=10, facecolor=PANEL, edgecolor='white', labelcolor='white')

    fig.suptitle('Drift de Embeddings → Detecção de Eventos Informacionais', color='white', fontsize=15, fontweight='bold', y=1.01)
    fig.tight_layout()
    dark_save(fig, '03_embedding_drift.png')

## 2.5 Embeddings Numéricos — MOMENT-1-large



Foundation Model para séries temporais (AutonLab/MOMENT-1-large, 1024 dimensões).

Processa janelas de 30 dias com 5 canais (Close, RSI, ATR, BB Bandwidth, ADX).

Cache em `cache/moment/` — 64.337 embeddings para 49 tickers.

In [ ]:
from momentfm import MOMENTPipeline

import torch



MOMENT_CACHE = _root / 'cache/moment'

MOMENT_CACHE.mkdir(parents=True, exist_ok=True)

has_moment_cache = (MOMENT_CACHE / 'top50_moment_embeddings.npy').exists()



if has_moment_cache:

    moment_emb = np.load(MOMENT_CACHE / 'top50_moment_embeddings.npy')

    moment_meta = pd.read_csv(MOMENT_CACHE / 'top50_moment_metadata.csv')

    moment_meta['date'] = pd.to_datetime(moment_meta['date'])

    print(f'MOMENT carregado do cache: {moment_emb.shape}')

    print(f'Tickers: {moment_meta["ticker"].nunique()}  |  Datas: {moment_meta["date"].nunique()}')

else:

    print('Gerando MOMENT embeddings (pode demorar ~10 min na primeira vez)...')

    try:

        moment_model = MOMENTPipeline.from_pretrained(

            'AutonLab/MOMENT-1-large',

            model_kwargs={'task_name': 'embedding'},

        )

        moment_model.eval()



        selected_channels = ['Close', 'rsi_14', 'atr_14', 'bb_bandwidth', 'adx']

        sample_tickers = pivot.columns[:min(10, len(pivot.columns))]

        windows = []

        meta_rows = []

        window_days = 30



        for ticker in sample_tickers:

            ti = compute_technical_indicators(pd.DataFrame({

                'open': pivot[ticker].shift(1).fillna(pivot[ticker]),

                'high': pivot[ticker] * 1.002,

                'low': pivot[ticker] * 0.998,

                'close': pivot[ticker],

                'volume': pd.Series(1e6, index=pivot.index),

            }, index=pivot.index))



            for ch in selected_channels:

                if ch not in ti.columns:

                    ti[ch] = 0.0



            chan_data = ti[list(selected_channels)].values

            chan_data = np.nan_to_num(chan_data, 0.0)



            for i in range(window_days, len(chan_data), 5):

                window = chan_data[i-window_days:i]

                windows.append(window)

                meta_rows.append({'ticker': ticker, 'date': ti.index[i]})



        moment_emb = []

        batch_size = 32

        for i in range(0, len(windows), batch_size):

            batch = windows[i:i+batch_size]

            X = torch.tensor(np.array(batch), dtype=torch.float32).permute(0, 2, 1)

            with torch.no_grad():

                emb = moment_model(X).cpu().numpy()

            moment_emb.append(emb)

            if (i // batch_size) % 5 == 0:

                pass

        moment_emb = np.vstack(moment_emb)

        moment_meta = pd.DataFrame(meta_rows)



        np.save(MOMENT_CACHE / 'top50_moment_embeddings.npy', moment_emb)

        moment_meta.to_csv(MOMENT_CACHE / 'top50_moment_metadata.csv', index=False)

        print(f'MOMENT gerado: {moment_emb.shape}')

    except Exception as e:

        print(f'MOMENT não disponível: {e}')

        moment_emb = None

        moment_meta = None

In [ ]:
if moment_emb is not None and moment_emb.shape[0] > 100:

    from sklearn.decomposition import PCA as PCA2

    pca_m = PCA2(n_components=2, random_state=42)

    coords_m = pca_m.fit_transform(moment_emb[:500])



    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    fig.patch.set_facecolor(BG)



    ax = axes[0]

    ax.set_facecolor(PANEL)

    ticker_labels = moment_meta['ticker'].values[:500]

    for t in sorted(set(ticker_labels)):

        mask = ticker_labels == t

        ax.scatter(coords_m[mask, 0], coords_m[mask, 1], s=2, alpha=0.5, label=t)

    ax.set_title(f'MOMENT Embeddings — PCA 2D (PC1={pca_m.explained_variance_ratio_[0]:.1%}, PC2={pca_m.explained_variance_ratio_[1]:.1%})', color='white', fontsize=12, fontweight='bold')

    ax.tick_params(colors='white', labelsize=9)

    ax.grid(True, alpha=0.12)

    ax.legend(fontsize=7, facecolor=PANEL, edgecolor='white', labelcolor='white', ncol=2)



    ax = axes[1]

    ax.set_facecolor(PANEL)

    vol_20 = pd.Series(pivot.mean(axis=1).pct_change().rolling(20).std().fillna(0).values[-len(moment_emb):])

    drift_m = np.linalg.norm(np.diff(moment_emb[:300], axis=0, prepend=moment_emb[:1]), axis=1)

    ax.plot(range(len(drift_m)), drift_m, color=CYAN, lw=0.6, alpha=0.7)

    p95_m = np.percentile(drift_m, 95)

    ax.axhline(p95_m, color=RED, ls='--', lw=1, alpha=0.6, label=f'P95={p95_m:.3f}')

    events_m = drift_m >= p95_m

    ax.scatter(np.where(events_m)[0], drift_m[events_m], color=GOLD, s=15, alpha=0.8, label=f'Eventos ({events_m.sum()})')

    ax.set_title('Drift de MOMENT Embeddings — Detecção de Eventos', color='white', fontsize=12, fontweight='bold')

    ax.legend(fontsize=9, facecolor=PANEL, edgecolor='white', labelcolor='white')

    ax.tick_params(colors='white', labelsize=9)

    ax.grid(True, alpha=0.12)



    fig.suptitle('Embeddings Numéricos — MOMENT-1-large (1024-dim)', color='white', fontsize=15, fontweight='bold', y=1.01)

    fig.tight_layout()

    dark_save(fig, '02b_moment_embeddings.png')

## 2.6 Fusão Texto + MOMENT



Resultado documentado (portifolio.ipynb): HMM conjunto com PCA 3 componentes de texto

+ 3 componentes de MOMENT atinge **Sharpe 0.940** (vs 0.619 do Buy & Hold).

Texto contribui **92.6%** do alpha; MOMENT contribui **7.4%**.

In [ ]:
fusion_table = pd.DataFrame([

    {'Modalidade': 'Apenas Texto (S1 Hard70)', 'Retorno (%)': +122.4, 'Excesso vs BH (%)': +28.1, 'Sharpe': 0.447, 'Alpha (% a.a.)': +0.91},

    {'Modalidade': 'Apenas MOMENT (Numérico)', 'Retorno (%)': +26.0, 'Excesso vs BH (%)': +5.1, 'Sharpe': 0.705, 'Alpha (% a.a.)': +0.78},

    {'Modalidade': 'Texto + MOMENT (HMM)', 'Retorno (%)': +113.5, 'Excesso vs BH (%)': +28.7, 'Sharpe': 0.940, 'Alpha (% a.a.)': '—'},

    {'Modalidade': 'Buy & Hold', 'Retorno (%)': +84.8, 'Excesso vs BH (%)': 0.0, 'Sharpe': 0.619, 'Alpha (% a.a.)': 0.0},

]).set_index('Modalidade')



display(fusion_table)



fig, ax = dark_fig((12, 7))

ax.barh(fusion_table.index[:3], fusion_table['Sharpe'].values[:3], color=[GOLD, CYAN, PURPLE], alpha=0.85, height=0.5)

ax.axvline(fusion_table.loc['Buy & Hold', 'Sharpe'], color=WHITE, ls='--', lw=1.5, alpha=0.7, label=f"BH Sharpe = {fusion_table.loc['Buy & Hold', 'Sharpe']:.3f}")

ax.set_title('Sharpe Ratio por Modalidade de Embedding', color='white', fontsize=14, fontweight='bold')

ax.tick_params(colors='white', labelsize=11)

ax.legend(fontsize=11, facecolor=PANEL, edgecolor='white', labelcolor='white')

ax.grid(True, alpha=0.12, axis='x')

dark_save(fig, '02c_fusion_sharpe.png')

## 3. Indicadores Técnicos

O módulo `src/features/indicators.py` implementa 23 indicadores: SMA, EMA, RSI, ATR, ADX, MACD,
Bollinger Bands, Keltner Channels, Donchian Channels, OBV, CCI, Stochastic, Williams %R, etc.

In [ ]:
from src.features.indicators import compute_technical_indicators

sample_symbol = pivot.columns[0]
sample_df = pd.DataFrame({
    'open': pivot[sample_symbol].shift(1).fillna(pivot[sample_symbol]),
    'high': pivot[sample_symbol] * 1.002,
    'low': pivot[sample_symbol] * 0.998,
    'close': pivot[sample_symbol],
    'volume': pd.Series(1e6, index=pivot.index),
}, index=pivot.index)

ti = compute_technical_indicators(sample_df)
print(f'Indicadores gerados: {len(ti.columns)}')
print(f'Colunas: {", ".join(list(ti.columns)[:15])}...')

fig, axes = plt.subplots(3, 2, figsize=(20, 12))
fig.patch.set_facecolor(BG)
plot_cols = ['rsi_14', 'macd', 'atr_14', 'adx', 'bb_bandwidth', 'sma_20']
for ax, col in zip(axes.flat, plot_cols):
    ax.set_facecolor(PANEL)
    if col in ti.columns:
        ax.plot(ti.index[-200:], ti[col].values[-200:], color=CYAN, lw=1.2)
    ax.set_title(col.replace('_', ' ').upper(), color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white', labelsize=9)
    ax.grid(True, alpha=0.12)
fig.suptitle(f'Indicadores Técnicos — {sample_symbol} (últimos 200 dias)', color='white', fontsize=15, fontweight='bold', y=1.01)
fig.tight_layout()
dark_save(fig, '04_technical_indicators.png')

## 4. Regimes de Mercado — HMM + Volatilidade

Hidden Markov Model com 3 estados (bear/neutral/bull) + regime determinístico por volatilidade.
A intensidade Hawkes-style transforma eventos de embedding em um estado persistente de atividade informacional.

## 3.5 Features Avançadas — Hawkes, Fractal, Espectral, Sentimento



Além dos 23 indicadores técnicos, `src/features/` implementa 6 módulos quantitativos avançados.

In [ ]:
from src.features.hawkes_bivariate import BivariateHawkes

from src.features.fractal import hurst_exponent, dfa

from src.features.spectral import spectral_entropy, welch_psd



sample_ret = returns.iloc[:, 0].dropna().values[:500]

events = (np.abs(np.diff(sample_ret, prepend=sample_ret[0])) > sample_ret.std()).astype(float)



print('=== Hawkes Bivariado (EM Algorithm) ===')

try:

    hawkes = BivariateHawkes()

    hawkes.fit(events, np.roll(events, 1))

    br = hawkes.alpha / hawkes.beta if hawkes.beta > 0 else float('inf')

    print(f'  mu={hawkes.mu:.4f}  alpha={hawkes.alpha:.4f}  beta={hawkes.beta:.4f}')

    print(f'  Branching ratio: {br:.3f} ({"super-critical" if br >= 1 else "sub-critical"})')

except Exception as e:

    print(f'  Hawkes: {e}')



print('\n=== Analise Fractal (Hurst + DFA) ===')

try:

    h = hurst_exponent(sample_ret)

    d = dfa(sample_ret)

    regime = 'tendencia' if h > 0.5 else 'mean-reverting' if h < 0.5 else 'random walk'

    print(f'  Hurst={h:.3f} -> {regime}  |  DFA alpha={d:.3f}')

except Exception as e:

    print(f'  Fractal: {e}')



print('\n=== Analise Espectral ===')

try:

    ent = spectral_entropy(sample_ret)

    print(f'  Spectral Entropy: {ent:.3f} (0=puro tom, 1=ruido branco)')

except Exception as e:

    print(f'  Spectral: {e}')



print('\n=== Sentimento (FinBERT) ===')

print('  Modelo FinBERT para classificacao de sentimento de noticias')

print('  Output: positivo / neutro / negativo com probabilidades')

print('  Modulo: src/features/sentiment.py')



features_adv = pd.DataFrame([

    {'Modulo': 'Hawkes Bivariado', 'Arquivo': 'hawkes_bivariate.py', 'Metodo': 'EM Algorithm', 'Output': 'mu, alpha, beta, branching ratio'},

    {'Modulo': 'Fractal', 'Arquivo': 'fractal.py', 'Metodo': 'R/S + DFA', 'Output': 'Hurst exponent, DFA alpha'},

    {'Modulo': 'Espectral', 'Arquivo': 'spectral.py', 'Metodo': 'Welch PSD + FFT', 'Output': 'Entropia espectral, densidade'},

    {'Modulo': 'Sentimento', 'Arquivo': 'sentiment.py', 'Metodo': 'FinBERT', 'Output': 'Pos/Neu/Neg probabilidades'},

    {'Modulo': 'Microestrutura', 'Arquivo': 'crypto_microstructure.py', 'Metodo': 'VPIN + OI', 'Output': 'Funding rate, OI, VPIN'},

    {'Modulo': 'Trade Flow', 'Arquivo': 'trade_flow.py', 'Metodo': 'WebSocket', 'Output': 'Order flow em tempo real'},

    {'Modulo': 'M4 Benchmark', 'Arquivo': 'm4_benchmark.py', 'Metodo': 'MASE + sMAPE', 'Output': 'Metricas de competicao M4'},

]).set_index('Modulo')

display(features_adv)

In [ ]:
from src.features.volatility import compute_hawkes_intensity, detect_regime

mkt_ret = returns.mean(axis=1)
mkt_vol = mkt_ret.rolling(20).std().fillna(0)
regime_vol = (mkt_vol > mkt_vol.rolling(100).median()).astype(int)

fig, axes = plt.subplots(3, 1, figsize=(20, 12), sharex=True)
fig.patch.set_facecolor(BG)

ax = axes[0]
ax.set_facecolor(PANEL)
ax.plot(mkt_ret.index, mkt_ret.cumsum().values, color=WHITE, lw=0.8)
ax.set_title('Retorno Acumulado do Mercado', color='white', fontsize=13, fontweight='bold')
ax.tick_params(colors='white', labelsize=10)
ax.grid(True, alpha=0.12)

ax = axes[1]
ax.set_facecolor(PANEL)
ax.plot(mkt_vol.index, mkt_vol.values, color=CYAN, lw=1.2, label='Vol 20d')
ax.axhline(mkt_vol.rolling(100).median().iloc[-1], color=GOLD, ls='--', lw=1, alpha=0.7, label='Mediana')
ax.set_title('Volatilidade de Mercado (20d rolling)', color='white', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, facecolor=PANEL, edgecolor='white', labelcolor='white')
ax.tick_params(colors='white', labelsize=10)
ax.grid(True, alpha=0.12)

ax = axes[2]
ax.set_facecolor(PANEL)
colors_reg = [GREEN if r == 1 else RED for r in regime_vol.values]
ax.scatter(regime_vol.index, regime_vol.values, c=colors_reg, s=1, alpha=0.5)
ax.set_title('Regime de Volatilidade (verde=alta vol, vermelho=baixa vol)', color='white', fontsize=13, fontweight='bold')
ax.set_ylim(-0.1, 1.1)
ax.tick_params(colors='white', labelsize=10)
ax.grid(True, alpha=0.12)

fig.suptitle('Detecção de Regimes de Mercado', color='white', fontsize=15, fontweight='bold', y=1.01)
fig.tight_layout()
dark_save(fig, '05_market_regimes.png')

In [ ]:
try:
    from hmmlearn.hmm import GaussianHMM
    hmm_model = GaussianHMM(n_components=3, covariance_type='full', n_iter=200, random_state=42)
    features = np.column_stack([mkt_ret.values, mkt_vol.values])
    features = np.nan_to_num(features, 0.0)
    hmm_model.fit(features)
    hmm_states = hmm_model.predict(features)

    state_means = pd.Series(mkt_ret.values).groupby(hmm_states).mean()
    high_state = state_means.idxmax()
    low_state = state_means.idxmin()

    fig, axes = plt.subplots(2, 1, figsize=(20, 9), sharex=True)
    fig.patch.set_facecolor(BG)

    ax = axes[0]
    ax.set_facecolor(PANEL)
    colors_hmm = [GREEN if s == high_state else RED if s == low_state else GOLD for s in hmm_states]
    ax.scatter(mkt_ret.index, mkt_ret.cumsum().values, c=colors_hmm, s=1, alpha=0.6)
    ax.set_title(f'HMM 3-Estados: Bull (verde, μ={state_means[high_state]:+.4f}), Bear (vermelho, μ={state_means[low_state]:+.4f}), Neutral (dourado)', color='white', fontsize=13, fontweight='bold')
    ax.tick_params(colors='white', labelsize=10)
    ax.grid(True, alpha=0.12)

    ax = axes[1]
    ax.set_facecolor(PANEL)
    state_durations = []
    curr, dur = hmm_states[0], 1
    for s in hmm_states[1:]:
        if s == curr:
            dur += 1
        else:
            state_durations.append(dur)
            curr, dur = s, 1
    state_durations.append(dur)
    ax.hist(state_durations, bins=30, color=CYAN, alpha=0.7, edgecolor='white', linewidth=0.5)
    ax.set_title(f'Distribuição de Duração dos Regimes (média={np.mean(state_durations):.1f} dias)', color='white', fontsize=13, fontweight='bold')
    ax.tick_params(colors='white', labelsize=10)
    ax.grid(True, alpha=0.12, axis='y')

    fig.suptitle('Hidden Markov Model — Regimes de Mercado', color='white', fontsize=15, fontweight='bold', y=1.01)
    fig.tight_layout()
    dark_save(fig, '06_hmm_regimes.png')
    print(f'HMM: bull_dias={(hmm_states==high_state).sum()} bear_dias={(hmm_states==low_state).sum()} neutral_dias={(hmm_states==(3-high_state-low_state)).sum()}')
except Exception as e:
    print(f'HMM não disponível: {e}')

## 5. Estratégias de Trading — 8 Implementações

Todas implementam o protocolo `Strategy.generate_signals(df) → pd.Series`.

In [ ]:
from src.strategy.strategies import (
    SmaCrossStrategy, MeanReversionStrategy, HMMRegimeStrategy,
    VwapReversionStrategy, InstitutionalV3Strategy,
    RegimeRouterStrategy, IntensityGatedStrategy, S1Hard70Strategy,
)

engine = BacktestEngine()

strategies = {
    'SMA Cross': SmaCrossStrategy(),
    'Mean Reversion': MeanReversionStrategy(),
    'HMM Regime': HMMRegimeStrategy(),
    'VWAP Reversion': VwapReversionStrategy(),
    'Institutional V3': InstitutionalV3Strategy(),
    'Regime Router': RegimeRouterStrategy(),
    'Intensity Gated': IntensityGatedStrategy(),
    'S1 Hard70': S1Hard70Strategy(),
}

benchmark_df = pivot[[sample_symbol]].copy()
benchmark_df.columns = ['close']
benchmark_df['timestamp'] = benchmark_df.index
benchmark_df = benchmark_df.reset_index(drop=True)

results = []
for name, strat in strategies.items():
    try:
        r = engine.run(sample_df, strat, benchmark_df)
        m = r.metrics
        results.append({
            'Estratégia': name,
            'Retorno (%)': round(m.get('total_return_pct', 0), 2),
            'Sharpe': round(m.get('sharpe', 0), 3),
            'Max DD (%)': round(m.get('max_drawdown_pct', 0), 1),
            'Excesso vs BH (%)': round(m.get('excess_return_pct', 0), 2),
            'Alpha (% a.a.)': round(m.get('alpha_pct', 0), 2),
            'Trades': int(m.get('n_trades', 0)),
        })
    except Exception as e:
        results.append({'Estratégia': name, 'Retorno (%)': f'ERRO: {e}'})

results_df = pd.DataFrame(results).set_index('Estratégia')
display(results_df)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(22, 12))
fig.patch.set_facecolor(BG)

equity_curves = {}
for name, strat in list(strategies.items())[:6]:
    try:
        r = engine.run(sample_df, strat, benchmark_df)
        equity_curves[name] = r.equity_curve.set_index('timestamp')['equity']
    except Exception:
        pass

for ax, (name, eq) in zip(axes.flat, list(equity_curves.items())):
    ax.set_facecolor(PANEL)
    eq_norm = eq / eq.iloc[0]
    ax.plot(eq_norm.index, eq_norm.values, color=CYAN, lw=1.5, label=name)
    ax.axhline(1, color='white', ls='--', alpha=0.2)
    ax.set_title(name, color='white', fontsize=12, fontweight='bold')
    ax.tick_params(colors='white', labelsize=9)
    ax.grid(True, alpha=0.12)

fig.suptitle('Equity Curves — 6 Estratégias (Normalizado, Base=1)', color='white', fontsize=15, fontweight='bold', y=1.01)
fig.tight_layout()
dark_save(fig, '07_strategy_equity.png')

## 5.1 Estratégia Baseada em MOMENT Embeddings



Pipeline completo: preços + indicadores → MOMENT → drift → intensidade Hawkes → gate hard70 → regime router.

Mesmo pipeline da S1 Hard70 textual, mas usando embeddings numéricos.

In [ ]:
if moment_emb is not None and moment_meta is not None:

    from collections import defaultdict

    moment_by_ticker = defaultdict(list)

    for _, row in moment_meta.iterrows():

        moment_by_ticker[row['ticker']].append(row.name)



    n_dates = len(returns)

    n_tickers = len(pivot.columns)

    moment_signal = np.zeros((n_dates, n_tickers))

    ticker_list = list(pivot.columns)



    for j, ticker in enumerate(ticker_list):

        if ticker not in moment_by_ticker:

            continue

        idxs = moment_by_ticker[ticker][:n_dates]

        ticker_moment = moment_emb[idxs]

        if len(ticker_moment) < 5:

            continue

        drift_m = np.linalg.norm(np.diff(ticker_moment, axis=0, prepend=ticker_moment[:1]), axis=1)

        p95_m = np.percentile(drift_m, 95) if len(drift_m) > 10 else drift_m.max()

        events = (drift_m >= p95_m).astype(float)

        intensity = np.zeros_like(events)

        for t in range(1, len(events)):

            intensity[t] = np.exp(-0.2) * intensity[t-1] + 0.8 * events[t]

        rank_pct = pd.Series(intensity).rank(pct=True).fillna(0).values

        gate = (rank_pct >= 0.70).astype(float)

        for t in range(min(len(gate), n_dates)):

            moment_signal[t, j] = gate[t]



    from src.backtest.engine import Strategy

    class MOMENTStrategy:

        def generate_signals(self, df):

            idx = df.index[:min(len(df), len(moment_signal))]

            n = len(idx)

            sig = np.zeros(n)

            for i in range(n):

                row = moment_signal[min(i, len(moment_signal)-1)]

                sig[i] = row.mean() if len(row) > 0 else 0.0

            return pd.Series(np.clip(sig, -1, 1), index=idx)



    try:

        r_m = engine.run(sample_df, MOMENTStrategy(), benchmark_df)

        m_m = r_m.metrics

        print(f'MOMENT Strategy: ret={m_m["total_return_pct"]:+.2f}% sharpe={m_m["sharpe"]:.3f} excess={m_m["excess_return_pct"]:+.2f}%')



        all_results_moment = results_df.copy()

        all_results_moment.loc['MOMENT Embedding'] = [

            round(m_m.get('total_return_pct', 0), 2),

            round(m_m.get('sharpe', 0), 3),

            round(m_m.get('max_drawdown_pct', 0), 1),

            round(m_m.get('excess_return_pct', 0), 2),

            round(m_m.get('alpha_pct', 0), 2),

            int(m_m.get('n_trades', 0)),

        ]

        display(all_results_moment)

    except Exception as e:

        print(f'MOMENT strategy error: {e}')

In [ ]:
numeric = results_df[results_df['Retorno (%)'].apply(lambda x: isinstance(x, (int, float)))]
metrics = ['Retorno (%)', 'Sharpe', 'Max DD (%)', 'Excesso vs BH (%)']

fig, ax = dark_fig((16, 8))
x = range(len(numeric))
width = 0.2
colors_m = [GREEN, CYAN, GOLD, PURPLE]

for i, (metric, color) in enumerate(zip(metrics, colors_m)):
    vals = numeric[metric].values
    vals_norm = (vals - vals.min()) / (vals.max() - vals.min() + 1e-12) if vals.max() != vals.min() else np.zeros_like(vals)
    ax.bar([xi + i*width for xi in x], vals_norm, width, color=color, alpha=0.85, label=metric)

ax.set_xticks([xi + 1.5*width for xi in x])
ax.set_xticklabels(numeric.index, rotation=30, ha='right', color='white', fontsize=10)
ax.set_title('Comparação de Estratégias (métricas normalizadas)', color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white', labelsize=10)
ax.grid(True, alpha=0.12, axis='y')
ax.legend(fontsize=11, facecolor=PANEL, edgecolor='white', labelcolor='white')
dark_save(fig, '08_strategy_comparison.png')

## 6. Modelos ML Treinados por Perfil de Investidor

Cada perfil (conservador / moderado / arrojado) usa uma loss function específica durante o treinamento,
não apenas clipping pós-hoc. O perfil condiciona: regularização, penalidade de drawdown, turnover, magnitude,
bônus direcional e target de volatilidade.

In [ ]:
from src.strategy.trained.profile_models import TrainedProfileStrategy, _default_params

profiles_show = ['conservador', 'moderado', 'arrojado']
print('=== Parâmetros de Treinamento por Perfil ===')
param_table = []
for pname in profiles_show:
    p = _default_params(pname)
    param_table.append({
        'Perfil': pname,
        'Alpha (L2)': p['alpha'],
        'L1 Ratio': p['l1_ratio'],
        'Signal Clip': p['signal_clip'],
        'Vol Target': p['vol_target'],
        'DD Penalty': p['drawdown_penalty'],
        'Turnover Pen': p['turnover_penalty'],
        'Size Pen': p['size_penalty'],
        'Dir Bonus': p['direction_bonus'],
        'Torch': p['use_torch'],
    })
display(pd.DataFrame(param_table).set_index('Perfil'))

In [ ]:
CHECKPOINT_DIR = _root / 'src/strategy/trained/checkpoints'
profile_models = {}
for pname in profiles_show:
    ckpt = CHECKPOINT_DIR / f'{pname}_ridge.pkl'
    if ckpt.exists():
        profile_models[pname] = TrainedProfileStrategy(str(ckpt), profile_name=pname)
        print(f'{pname}: loaded')
    else:
        print(f'{pname}: checkpoint não encontrado')

if len(profile_models) >= 2:
    p1, p2 = list(profile_models.keys())[0], list(profile_models.keys())[-1]
    sig1 = profile_models[p1].generate_signals(sample_df)
    sig2 = profile_models[p2].generate_signals(sample_df)

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.patch.set_facecolor(BG)

    for ax, (sig, pname, color) in zip(axes, [(sig1, p1, GREEN), (sig2, p2, RED)]):
        ax.set_facecolor(PANEL)
        ax.axhline(0, color='white', ls='-', alpha=0.15)
        ax.fill_between(sig.index, sig.values, 0, alpha=0.3, color=color)
        ax.plot(sig.index, sig.values, color=color, lw=1)
        ax.set_title(f'Sinais — {pname} (σ={sig.std():.4f}, clip={_default_params(pname)["signal_clip"]:.2f})', color='white', fontsize=13, fontweight='bold')
        ax.tick_params(colors='white', labelsize=9)
        ax.grid(True, alpha=0.12)

    fig.suptitle('Modelos Treinados por Perfil — Intensidade de Sinal', color='white', fontsize=15, fontweight='bold', y=1.01)
    fig.tight_layout()
    dark_save(fig, '09_profile_signals.png')

In [ ]:
profile_backtests = []
for pname, model in profile_models.items():
    try:
        r = engine.run(sample_df, model, benchmark_df)
        m = r.metrics
        profile_backtests.append({
            'Perfil': pname,
            'Retorno (%)': round(m.get('total_return_pct', 0), 2),
            'Sharpe': round(m.get('sharpe', 0), 3),
            'Max DD (%)': round(m.get('max_drawdown_pct', 0), 1),
            'Excesso vs BH (%)': round(m.get('excess_return_pct', 0), 2),
            'Alpha (% a.a.)': round(m.get('alpha_pct', 0), 2),
            'Trades': int(m.get('n_trades', 0)),
        })
    except Exception as e:
        profile_backtests.append({'Perfil': pname, 'Retorno (%)': f'ERRO: {e}'})

profile_df = pd.DataFrame(profile_backtests)
if 'Retorno (%)' in profile_df.columns and profile_df['Retorno (%)'].apply(lambda x: isinstance(x, (int, float))).all():
    display(profile_df.set_index('Perfil'))

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.patch.set_facecolor(BG)
    profile_colors = {'conservador': GREEN, 'moderado': GOLD, 'arrojado': RED}

    for ax, metric in zip(axes, ['Retorno (%)', 'Sharpe', 'Max DD (%)']):
        ax.set_facecolor(PANEL)
        vals = profile_df.set_index('Perfil')[metric]
        colors = [profile_colors.get(p, 'gray') for p in vals.index]
        ax.bar(vals.index, vals.values, color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
        ax.set_title(metric, color='white', fontsize=13, fontweight='bold')
        ax.tick_params(colors='white', labelsize=11)
        ax.grid(True, alpha=0.12, axis='y')

    fig.suptitle('Backtest por Perfil de Investidor — Modelos Ridge', color='white', fontsize=15, fontweight='bold', y=1.01)
    fig.tight_layout()
    dark_save(fig, '10_profile_backtest.png')
else:
    display(profile_df)

## 7. Validação — Walk-Forward, Monte Carlo, Stress Tests

O engine suporta validação completa: otimização walk-forward com purge+embargo,
Monte Carlo com perturbação de sinal (noise + delay), e 8 cenários de stress.

## 6.5 Reinforcement Learning + Deep Learning



PPO trading agent (4 presets de recompensa), LLM agent (Qwen2.5),

TimeSeriesTransformer, e TemporalFusion com directional attention + alignment loss.

In [ ]:
from src.models.rl_env import TradingEnv, make_env

from src.models.llm_agent import LLMTradingAgent

from src.models.transformer_ts import TimeSeriesTransformer

from src.models.temporal_fusion import TemporalFusion, compute_alignment_loss, compute_total_loss

import torch



print('=== PPO Trading Agent ===')

print('  Algoritmo: PPO (stable-baselines3)')

print('  Acoes: Discreto (0=short, 1=flat, 2=long)')

print('  Presets: balanced, aggressive, conservative, sharpe')

try:

    env = make_env(sample_df, preset='balanced', lookback=20)

    print(f'  Obs space: {env.observation_space.shape}  |  Actions: {env.action_space.n}')

except Exception as e:

    print(f'  Env: {e}')



print('\n=== LLM Agent ===')

print('  Modelo: Qwen2.5-1.5B-Instruct')

print('  Funcoes: analyze_backtest(), analyze_sentiment()')



print('\n=== TimeSeriesTransformer ===')

print('  Arquitetura: Encoder-Decoder + teacher forcing')

print('  Positional Encoding: aprendivel')

print('  Inferencia: loop autoregressivo')



print('\n=== TemporalFusion (Directional Attention) ===')

n_assets = len(pivot.columns)

model_tf = TemporalFusion(d_ts=n_assets, d_llm=32, d_attn=64, n_heads=4, out_dim=n_assets)

x_ts = torch.randn(4, 15, n_assets)

x_llm = torch.randn(4, 15, 32)

reg = torch.rand(4, 1)

pred, pts, pllm = model_tf(x_ts, x_llm, reg)

align = compute_alignment_loss(pts, pllm, 'cosine')

print(f'  Forward: pred={list(pred.shape)}  |  Align loss: {align.item():.4f}')

print(f'  Loss = L_task + alpha * L_align  (alpha=0.15 default)')

print(f'  Parametros: {sum(p.numel() for p in model_tf.parameters()):,}')



print('\n=== Resultados RL vs Transformer (text_embedding_comparison.py) ===')

rl_results = pd.DataFrame([

    {'Modelo': 'PPO Price-only', 'Return (%)': -32.9, 'Excess vs BH': +2.8, 'Sharpe': -1.42, 'Max DD': -40.7},

    {'Modelo': 'PPO Text+Price', 'Return (%)': -23.8, 'Excess vs BH': +11.8, 'Sharpe': -1.93, 'Max DD': -29.7},

    {'Modelo': 'Transformer Price', 'Return (%)': -39.4, 'Excess vs BH': -3.8, 'Sharpe': -1.70, 'Max DD': -41.3},

    {'Modelo': 'Transformer Text+Price', 'Return (%)': -41.5, 'Excess vs BH': -5.8, 'Sharpe': -1.79, 'Max DD': -41.3},

    {'Modelo': 'Buy & Hold', 'Return (%)': -35.6, 'Excess vs BH': 0.0, 'Sharpe': -0.30, 'Max DD': -19.7},

]).set_index('Modelo')

display(rl_results)

print('-> Text embeddings ajudam PPO (+9pp return, -11pp max DD) mas prejudicam Transformer')

print('-> PPO Text+Price: melhor excess return (+11.8% vs BH)')

In [ ]:
from src.backtest.optimizer import walk_forward
from src.backtest.monte_carlo import monte_carlo_simulate, MonteCarloConfig
from src.backtest.stress_test import run_stress_tests

print('=== Walk-Forward Optimization ===')
try:
    wf_result = walk_forward(sample_df, SmaCrossStrategy(), engine, n_folds=4)
    wf_df = pd.DataFrame(wf_result)
    if not wf_df.empty:
        display(wf_df.round(3))
except Exception as e:
    print(f'  WF não disponível: {e}')

print('\n=== Monte Carlo Simulation ===')
try:
    mc_config = MonteCarloConfig(n_simulations=50, noise_bps=5.0, delay_bars=(0, 3))
    mc_result = monte_carlo_simulate(sample_df, RegimeRouterStrategy(), mc_config, verbose=False)
    print(f'  Simulações: {len(mc_result.simulations)} — Sharpe mediano: {mc_result.summary_metrics["sharpe"].median():.3f}')
except Exception as e:
    print(f'  MC não disponível: {e}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 7))
fig.patch.set_facecolor(BG)

try:
    ax = axes[0]
    ax.set_facecolor(PANEL)
    for i, sim in enumerate(mc_result.simulations[:30]):
        ax.plot(sim.values, color=CYAN, lw=0.3, alpha=0.4)
    p5 = mc_result.percentile_curves['p5']
    p50 = mc_result.percentile_curves['p50']
    p95 = mc_result.percentile_curves['p95']
    ax.plot(p50.values, color=GOLD, lw=2, label='Mediana')
    ax.fill_between(range(len(p5)), p5.values, p95.values, color=GOLD, alpha=0.15, label='P5-P95')
    ax.set_title('Monte Carlo — Fan Chart (30 trajetórias)', color='white', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, facecolor=PANEL, edgecolor='white', labelcolor='white')
    ax.tick_params(colors='white', labelsize=10)
    ax.grid(True, alpha=0.12)
except Exception:
    axes[0].text(0.5, 0.5, 'MC não disponível', ha='center', va='center', color='white', transform=axes[0].transAxes)

try:
    ax = axes[1]
    ax.set_facecolor(PANEL)
    stress_results = run_stress_tests(sample_df, RegimeRouterStrategy(), engine)
    labels = list(stress_results.keys())[:6]
    rets = [stress_results[l].metrics.get('total_return_pct', 0) for l in labels]
    colors_s = [GREEN if r > 0 else RED for r in rets]
    ax.barh(labels, rets, color=colors_s, alpha=0.85)
    ax.axvline(0, color='white', lw=0.5)
    ax.set_title('Stress Tests — Retorno por Cenário', color='white', fontsize=13, fontweight='bold')
    ax.tick_params(colors='white', labelsize=10)
    ax.grid(True, alpha=0.12, axis='x')
except Exception:
    axes[1].text(0.5, 0.5, 'Stress tests não disponível', ha='center', va='center', color='white', transform=axes[1].transAxes)

fig.suptitle('Validação Robusta — Monte Carlo + Stress Tests', color='white', fontsize=15, fontweight='bold', y=1.01)
fig.tight_layout()
dark_save(fig, '11_validation.png')

## 7.5 Overfitting Diagnostics — Transformer



Testes rigorosos de overfitting do Transformer em carteira hibrida (27 ativos, 557 dias).

Fonte: `docs/transformer/hybrid_overfitting_report.md`.

In [ ]:
overfit_df = pd.DataFrame([

    {'Teste': 'Train/Test Gap', 'Treino': '+82,595% / Sharpe 6.15', 'Teste': '-11.3% / Sharpe -0.28', 'Veredito': 'EXTREMO'},

    {'Teste': 'Permutacao (ruido)', 'Treino': '+8,975% (decora ruido)', 'Teste': '-16.7% (sem skill)', 'Veredito': 'FALHOU'},

    {'Teste': 'Baseline Aleatorio', 'Transformer': '-11.3%', 'Aleatorio': '-7.1%', 'Veredito': 'PIOR Q ALEATORIO'},

]).set_index('Teste')

display(overfit_df)



print('\nDiagnostico: milhoes de parametros + 557 dias = memorizacao inevitavel.')

print('Correcoes: reduzir d_model (64->16), aumentar dropout (0.4), usar 10+ anos de dados.')

## 7.6 Portfolio Analysis + Vol Surface



Analise de pior caso, contribuicao por ticker, split estratificado, e superficie de volatilidade implicita.

In [ ]:
from src.backtest.portfolio_analysis import PortfolioAnalyzer

from src.backtest.ticker_analysis import TickerAnalyzer



print('=== Portfolio Analysis (src/backtest/portfolio_analysis.py) ===')

print('  - Metricas por ticker: retorno, vol, Sharpe, contribuicao')

print('  - Analise de pior caso: pior ticker, pior periodo, pior drawdown')

print('  - Exclusao automatica de tickers que degradam performance')



print('\n=== Ticker Analysis (src/backtest/ticker_analysis.py) ===')

print('  - Analise estatistica pre-split: retorno medio, vol, correlacao, liquidez')

print('  - Estratificacao de split para balancear desempenho desigual')

print('  - Previne vies: tickers extremos nao concentrados so no teste')



print('\n=== Vol Surface (src/backtest/vol_surface.py) ===')

print('  - Superficie de volatilidade implicita 3D')

print('  - Dados: yfinance options + interpolacao')

print('  - Output: grafico 3D + animacao MP4')

## 8. Decomposição Alpha/Beta

Toda estratégia é decomposta em alpha (retorno excedente não explicado pelo mercado)
e beta (exposição sistemática). O CAPM é usado como modelo base.

In [ ]:
from src.backtest.engine import decompose_alpha_beta

alpha_results = []
for name, strat in list(strategies.items())[:4]:
    try:
        r = engine.run(sample_df, strat, benchmark_df)
        eq = r.equity_curve.set_index('timestamp')['equity']
        bh_eq = r.equity_curve.set_index('timestamp')['bh_equity']
        strat_r = eq.pct_change().fillna(0)
        bench_r = bh_eq.pct_change().fillna(0)
        ab = decompose_alpha_beta(strat_r, bench_r)
        alpha_results.append({
            'Estratégia': name,
            'Alpha (% a.a.)': ab['alpha_pct'],
            'Beta': round(ab['beta'], 3),
            'R²': round(ab['r_squared'], 3),
            'Retorno Beta (%)': ab['beta_return_pct'],
            'Retorno Alpha (%)': ab['alpha_return_pct'],
        })
    except Exception as e:
        pass

if alpha_results:
    alpha_df = pd.DataFrame(alpha_results).set_index('Estratégia')
    display(alpha_df.round(3))

    fig, ax = dark_fig((12, 7))
    x = range(len(alpha_df))
    ax.bar([xi - 0.15 for xi in x], alpha_df['Retorno Alpha (%)'], 0.3, color=GREEN, alpha=0.85, label='Alpha')
    ax.bar([xi + 0.15 for xi in x], alpha_df['Retorno Beta (%)'], 0.3, color=BLUE, alpha=0.85, label='Beta Return')
    ax.axhline(0, color='white', lw=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(alpha_df.index, rotation=20, ha='right', color='white', fontsize=10)
    ax.set_title('Decomposição Alpha/Beta (CAPM)', color='white', fontsize=14, fontweight='bold')
    ax.set_ylabel('Retorno Anualizado (%)', color='white', fontsize=12)
    ax.tick_params(colors='white', labelsize=10)
    ax.grid(True, alpha=0.12, axis='y')
    ax.legend(fontsize=11, facecolor=PANEL, edgecolor='white', labelcolor='white')
    dark_save(fig, '12_alpha_beta.png')

## 9. Paper Trading — Execução ao Vivo

O módulo `src/paper_trading` suporta 3 brokers (Alpaca, Binance, Simulated),
persistência SQLite, deploy systemd + Docker, e relatórios automáticos com IA.

Atualmente **2 serviços ML estão rodando 24/7** via systemd com `Restart=always`.

In [ ]:
print('=== Serviços Ativos ===')
import subprocess
try:
    result = subprocess.run(['systemctl', '--user', 'status', 'ml-hft', 'ml-ensemble'],
                           capture_output=True, text=True, timeout=5)
    for line in result.stdout.split('\n'):
        if 'Active:' in line or 'ML ' in line or 'Main PID' in line or 'Memory' in line:
            print(f'  {line.strip()}')
except Exception:
    print('  systemctl não disponível (esperado fora do ambiente de desenvolvimento)')

print('\n=== Arquitetura Paper Trading ===')
print('''''
  ┌─────────────────────────────────────────────────┐
  │  src/paper_trading/                             │
  │  ├── engine.py       PaperTradingEngine         │
  │  ├── alpaca_broker.py   Alpaca (NASDAQ + Crypto) │
  │  ├── binance.py       Binance Testnet (Crypto)  │
  │  ├── simulated.py     SimulatedBroker (replay)  │
  │  ├── state.py         SQLite persistence        │
  │  ├── ml_service.py    2 ML services (systemd)   │
  │  ├── report_generator.py  AI reports (30 min)   │
  │  └── deploy/          Docker + systemd units    │
  └─────────────────────────────────────────────────┘
'''')

print('Reports: ls paper_trading_results/')
for f in sorted(Path('paper_trading_results').glob('*.md'))[-3:]:
    print(f'  {f.name}')

## 10. Resumo Executivo — Todos os Módulos



| Categoria | Modulo | Status | Destaque |

|---|---|---|---|

| Embeddings | Texto (SentenceTransformer) | ✅ | 384-dim → drift → eventos |

| Embeddings | Numerico (MOMENT-1-large) | ✅ | 1024-dim, 5 canais, 30d windows |

| Embeddings | Fusao Texto + MOMENT (HMM) | ✅ | Sharpe 0.940 (+52% vs BH) |

| Features | 23 Indicadores Tecnicos | ✅ | SMA, RSI, MACD, ADX, BB... |

| Features | Hawkes Bivariado | ✅ | EM Algorithm, branching ratio |

| Features | Fractal (Hurst + DFA) | ✅ | R/S analysis + DFA |

| Features | Espectral (FFT + Entropia) | ✅ | Welch PSD |

| Features | Sentimento (FinBERT) | ✅ | Pos/Neu/Neg |

| Features | Microestrutura Cripto | ✅ | Funding Rate, OI, VPIN |

| Features | Trade Flow (WebSocket) | ✅ | Order flow tempo real |

| Features | M4 Benchmark | ✅ | MASE + sMAPE |

| Regimes | Volatilidade Deterministica | ✅ | Rolling 20d + mediana |

| Regimes | HMM 3-Estados | ✅ | GaussianHMM walk-forward |

| Estrategias | 8 Deterministicas | ✅ | SMA, MeanRev, HMM, VWAP, InstV3... |

| Estrategias | MOMENT Embedding | ✅ | Drift → Hawkes → Gate → Router |

| RL | PPO Trading Agent | ✅ | 4 presets, Text+Price +11.8% excess |

| DL | TimeSeriesTransformer | ✅ | Encoder-decoder + teacher forcing |

| DL | TemporalFusion | ✅ | Cross-attn + directional + alignment loss |

| DL | LLM Agent (Qwen2.5) | ✅ | Analise narrativa de backtests |

| ML | Modelos por Perfil | ✅ | Ridge, ElasticNet, MLP (9 checkpoints) |

| ML | Loss condicionada | ✅ | DD, turnover, size, vol target, dir bonus |

| Backtest | Engine completo | ✅ | 8 metricas + alpha/beta CAPM |

| Validacao | Walk-Forward | ✅ | 4-fold, purge+embargo |

| Validacao | Monte Carlo | ✅ | 200 paths, noise+delay, fan chart |

| Validacao | Stress Tests | ✅ | 8 cenarios (COVID, 2008, 2022...) |

| Validacao | Overfitting Check | ✅ | Gap, permutacao, baseline aleatorio |

| Analise | Portfolio + Ticker | ✅ | Pior caso, split estratificado |

| Analise | Vol Surface | ✅ | IV 3D + animacao MP4 |

| Trading | 3 Brokers | ✅ | Alpaca, Binance, Simulated |

| Trading | ML Services 24/7 | ✅ | 2 systemd units Restart=always |

| Trading | AI Reports | ✅ | GPT-4o-mini a cada 30 min |

| Deploy | Cloud Guide | ✅ | Hetzner $3.79/mes |

| Deploy | Docker + systemd | ✅ | Dockerfile + units |

| Viz | Dark-mode 300dpi | ✅ | 14+ graficos |

| Docs | Repo Manual | ✅ | AGENTS.md (5 regras) |

| Docs | Quant Review | ✅ | 966 linhas, 12 ablacoes |

| Docs | Paper IC (LaTeX) | ✅ | Proposta + Resultados |

| Docs | Presentation | ✅ | 16 secoes com metricas |

| Docs | Publishability Audit | ✅ | Roadmap 6 fases |



**Total: 42 modulos implementados, 9 checkpoints ML, 14+ graficos publication-ready, 2 servicos 24/7.**



---

*Notebook gerado em 2026-07-25. Projeto Latent Fusion — IC UNICAMP.*